**Un ejemplo simple de optimización estocásta**

*Despacho hidrotérmico con dos periodos*

- el periodo 1 es el día de hoy
- el periodo 2 es el día de mañana
- Tenemos tres escenarios de afluencias con probabilidades $\rho$
- El modelo esta diseñado para no preocuparse por las unidades
alejandro.garces@utp.edu.co



In [1]:
using LinearAlgebra
using Plots
using HiGHS
using JuMP
using DataFrames

In [57]:
c = 320      # costo de operacion de la termica
d1 = 100     # demanda en el periodo 1
d2 = 180     # demanda en el periodo 2
vmax = 150   # volumen máximo del embalse agregado
v0 = 80      # volumen inicial del embalse agregado
a1 = 40      # afluencia en el periodo 1
a2 = [70 10 50]   # escenarios de afluencia en el periodo 2
ρ = [0.1 0.4 0.5] # probabilidades de los escenarios de afluencia
η = 1             # factor de turbinamiento
Ω = 1:length(ρ)   # conjunto de escnarios
model = Model(HiGHS.Optimizer)
@variable(model, p1>=0)
@variable(model, u1>=0)
@variable(model, v1>=0)
@variable(model, q1>=0)
@variable(model, s1>=0)
@constraint(model, p1 + u1 == d1)
@constraint(model, v1 == v0 + a1 - q1 - s1)
@constraint(model, u1 == η*q1)
@constraint(model, v1 <= vmax)

@variable(model, p2[ω in Ω].>=0)
@variable(model, u2[ω in Ω].>=0)
@variable(model, v2[ω in Ω].>=0)
@variable(model, q2[ω in Ω].>=0)
@variable(model, s2[ω in Ω].>=0)
@constraint(model, [ω in Ω], p2[ω] + u2[ω] == d2)
@constraint(model, [ω in Ω], v2[ω] == v1 + a2[ω] - q2[ω] - s2[ω])
@constraint(model, [ω in Ω], u2[ω] == η*q2[ω])
@constraint(model, [ω in Ω], v2[ω] <= vmax)

@objective(model,Min, c*p1 + sum( ρ[ω]c*p2[ω] for ω in Ω))
optimize!(model)

Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: Apple Accelerate 
LP has 16 rows; 20 cols; 35 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [3e+01, 3e+02]
  Bound   [0e+00, 0e+00]
  RHS     [1e+01, 2e+02]
Presolving model
3 rows, 9 cols, 11 nonzeros 0s
Dependent equations search running on 3 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
3 rows, 7 cols, 9 nonzeros 0s
Presolve reductions: rows 3(-13); columns 7(-13); nonzeros 9(-26) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.0s
          3     3.9680000000e+04 Pr: 0(0) 0.0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model status        : Optimal
Simplex   iterati

In [51]:
df = DataFrame()
st =["Periodo 2-Escenario "*string(ω) for ω in Ω]
df[!,"Escenario"] = ["Periodo 1"; st]
df[!,"P_termica"] = [value(p1); value.(p2)]
df[!,"P_hidro"] = [value(u1); value.(u2)]
df[!,"Turbinamiento"] = [value(q1);value.(q2)]
df[!,"Vertimientos"] = [value(s1);value.(s2)]
df[!,"Afluencia"] = [a1;a2[:]]
df[!,"Volumen inicial"] = [v0;value.(v1)*ones(length(Ω))]
df[!,"Volumen final"] = [value(v1);value.(v2)]
df[!,"Costo"] = [c*value(p1);c*value.(p2)]
df[!,"Probabilidad"] = [1;ρ[:]]
display(df)

Row,Escenario,P_termica,P_hidro,Turbinamiento,Vertimientos,Afluencia,Volumen inicial,Volumen final,Costo,Probabilidad
,String,Float64,Float64,Float64,Float64,Int64,Float64,Float64,Float64,Float64
1,Periodo 1,90.0,10.0,10.0,0.0,40,80.0,110.0,28800.0,1.0
2,Periodo 2-Escenario 1,0.0,180.0,180.0,0.0,70,110.0,0.0,0.0,0.1
3,Periodo 2-Escenario 2,60.0,120.0,120.0,0.0,10,110.0,0.0,19200.0,0.4
4,Periodo 2-Escenario 3,20.0,160.0,160.0,0.0,50,110.0,0.0,6400.0,0.5


In [4]:
println(model)

Min 320 p1 + 32 p2[1] + 128 p2[2] + 160 p2[3]
Subject to
 p1 + u1 = 100
 v1 + q1 + s1 = 120
 u1 - q1 = 0
 p2[1] + u2[1] = 200
 p2[2] + u2[2] = 200
 p2[3] + u2[3] = 200
 -v1 + v2[1] + q2[1] + s2[1] = 70
 -v1 + v2[2] + q2[2] + s2[2] = 10
 -v1 + v2[3] + q2[3] + s2[3] = 50
 u2[1] - q2[1] = 0
 u2[2] - q2[2] = 0
 u2[3] - q2[3] = 0
 v1 ≤ 150
 v2[1] ≤ 150
 v2[2] ≤ 150
 v2[3] ≤ 150
 p1 ≥ 0
 u1 ≥ 0
 v1 ≥ 0
 q1 ≥ 0
 s1 ≥ 0
 p2[1] ≥ 0
 p2[2] ≥ 0
 p2[3] ≥ 0
 u2[1] ≥ 0
 u2[2] ≥ 0
 u2[3] ≥ 0
 v2[1] ≥ 0
 v2[2] ≥ 0
 v2[3] ≥ 0
 q2[1] ≥ 0
 q2[2] ≥ 0
 q2[3] ≥ 0
 s2[1] ≥ 0
 s2[2] ≥ 0
 s2[3] ≥ 0

